In [0]:
import os, random, sys, uuid
from datetime import datetime, timezone
sys.path.append(os.path.abspath('..'))

from src.config import CATALOGO, SPREAD_CORRETORA, TAXA_CORRETAGEM_PCT
from src.raw_writer import gravar_eventos

dbutils.widgets.text('operacoes', '20')
n_operacoes = int(dbutils.widgets.get('operacoes'))

SQL_MOEDAS = f'''
  SELECT c.code AS ativo,
         CASE WHEN c.code IN ('BTC', 'ETH') THEN 'CRIPTO' ELSE 'MOEDA' END AS classe,
         CAST(c.bid AS DOUBLE) AS bid,
         CAST(c.ask AS DOUBLE) AS ask,
         CAST(coletado_em_utc AS TIMESTAMP) AS coletado_em
  FROM {CATALOGO}.bronze.awesomeapi_cotacoes_atuais
  LATERAL VIEW explode(from_json(payload,
    'map<string, struct<code: string, bid: string, ask: string>>')) t AS par, c
'''

SQL_ACOES = f'''
  SELECT r.symbol AS ativo,
         'RENDA_VARIAVEL' AS classe,
         r.regularMarketPrice AS bid,
         r.regularMarketPrice AS ask,
         CAST(coletado_em_utc AS TIMESTAMP) AS coletado_em
  FROM {CATALOGO}.bronze.brapi_cotacoes
  LATERAL VIEW explode(from_json(payload,
    'struct<results: array<struct<symbol: string, regularMarketPrice: double>>>').results) t AS r
  WHERE r.symbol NOT LIKE '^%'
'''

partes = []
if spark.catalog.tableExists(f'{CATALOGO}.bronze.awesomeapi_cotacoes_atuais'):
    partes.append(SQL_MOEDAS)
if spark.catalog.tableExists(f'{CATALOGO}.bronze.brapi_cotacoes'):
    partes.append(SQL_ACOES)
tabela_clientes = f'{CATALOGO}.bronze.clientes'
if not partes or not spark.catalog.tableExists(tabela_clientes):
    dbutils.notebook.exit('Ainda não há cotações ou clientes na Bronze')

uniao = ' UNION ALL '.join(partes)
cotacoes = spark.sql(f'''
  SELECT ativo, classe, bid, ask
  FROM ({uniao})
  WHERE bid > 0
  QUALIFY ROW_NUMBER() OVER (PARTITION BY ativo ORDER BY coletado_em DESC) = 1
''').collect()

clientes = [r.id_cliente for r in spark.table(tabela_clientes)
            .filter("tipo_evento = 'CADASTRO'").select('id_cliente').distinct().collect()]

agora = datetime.now(timezone.utc).isoformat()
eventos = []

for _ in range(n_operacoes):
    cotacao = random.choice(cotacoes)
    lado = random.choices(['COMPRA', 'VENDA'], weights=[0.6, 0.4])[0]
    preco_mercado = cotacao.ask if lado == 'COMPRA' else cotacao.bid
    fator = 1 + SPREAD_CORRETORA if lado == 'COMPRA' else 1 - SPREAD_CORRETORA
    preco_executado = preco_mercado * fator
    valor_alvo = random.uniform(100, 20_000)

    if cotacao.classe == 'RENDA_VARIAVEL':
        quantidade = max(1, int(valor_alvo // preco_executado))
    elif cotacao.classe == 'CRIPTO':
        quantidade = round(valor_alvo / preco_executado, 8)
    else:
        quantidade = round(valor_alvo / preco_executado, 2)

    valor_financeiro = round(quantidade * preco_executado, 2)
    eventos.append({
        'id_operacao': str(uuid.uuid4()),
        'id_cliente': random.choice(clientes),
        'ativo': cotacao.ativo,
        'classe_ativo': cotacao.classe,
        'lado': lado,
        'quantidade': quantidade,
        'preco_mercado': round(preco_mercado, 8),
        'preco_executado': round(preco_executado, 8),
        'valor_financeiro': valor_financeiro,
        'taxa_corretagem': round(valor_financeiro * TAXA_CORRETAGEM_PCT, 2),
        'receita_spread': round(abs(preco_executado - preco_mercado) * quantidade, 2),
        'canal': random.choice(['APP', 'WEB', 'ASSESSOR']),
        'status': 'EXECUTADA',
        'executada_em_utc': agora,
    })

arquivo = gravar_eventos('operacoes', eventos)
print(f'{len(eventos)} operações gravadas em {arquivo}')